# 07 · Enhancement 3: LoRA (E3), from the go/no-go to the pre-registered analysis
Plan: `docs/LORA.md`; statistics pre-registered in `docs/LORA_ANALYSIS_PLAN.md` (plan v2); how to run:
`docs/ENHANCEMENTS_RUNBOOK.md` §3b. A Colab version with Drive checkpoints is `07_lora_colab`.

| Part | What | Needs | Time (T4, DA3-Small) |
|---|---|---|---|
| A · platform check | unit tests; with the **released** weights on the GPU: LoRA off = bit-identical, B = 0 = bit-identical; measured memory / time of one training step | nothing | ~10 min |
| B · KITTI conversion | raw drives -> canonical format, LiDAR depth (evaluation only), pre-registered drive split (Eigen-test leakage guard); only the drives of the split are converted | KITTI raw | ~1-5 min / drive |
| C · go/no-go | released DA3 unposed vs conditioned on the OXTS poses on the validation drives: **GO if the posed depth is ≥ 5% better** | B | ~10 min |
| D · fountain reference | released-model reconstruction of the fountain photos ("step 0") | photos | ~5 min |
| E · training | LoRA arms `e3`, `pose_only`, `own_output` × seeds, one complete seed after the other (2000 steps each, adapters + snapshots every 500 steps) | B, C = GO | ~1 h / run (part A prints the measured s/step) |
| F · KITTI test | released + every adapter on the held-out test drives, identical clips | E | ~15 min |
| G · Sec. 6 no-regression | released + E3 adapters on HiRoom, 7Scenes, DTU-64 (pose + reconstruction), the paper's protocol | E | 2-4 h / run: split `SEC6_RUNS` over sessions or platforms |
| H · fountain comparison | every final adapter and the snapshots of `SNAPSHOT_RUN` on the same photos: differences, side-by-side video, aligned PLYs | D, E | ~5 min / model |
| I · analysis | the pre-registered tests (docs/LORA_ANALYSIS_PLAN.md, plan v2) -> `analysis/summary.md` + result records | F (+ G) | 1 min |

**Inputs.** KITTI raw: register at cvlibs.net (KITTI → Raw Data) and accept the terms; with
`KITTI_RAW = "download"` part B then fetches only the ~25 drives of the pre-registered split from the
KITTI servers (image_02, oxts, velodyne only; no manual download). Public Kaggle "KITTI" sets checked on
2026-11-25 lack the poses (oxts) and are not usable. Plus the fountain photos.

**Sessions (suggested, 4-day plan).** 1: A-F, H, I.
2: G with `SEC6_RUNS = ["released"]` on one platform and `["e3_s0", "e3_s1"]` on the other, both resuming
from session 1's output; then H and I. Records of different sessions combine (same protocol).

Settings: **Accelerator = GPU (T4 / P100)**, **Internet = on**. Attach KITTI (zips or a KITTI raw
mirror) and the photos (right panel > Add Input), paste the paths with *Copy path*, choose the parts in
`RUN` (part G is off by default), then **Save Version → Save & Run All**.

**Resuming.** Every part, training run and Sec. 6 scene is skipped when its result exists; training
stops cleanly before the 12-h limit (`SESSION_H`). Next session: attach the previous version's output
(Add Input > Your Work) and set `RESUME_FROM`; `/tmp` (converted drives, cache) is rebuilt automatically.

In [ ]:
# --- repo setup (identical in every notebook) ---------------------------------------------
REPO_URL = "https://github.com/davideCabitz/DA3_LAB_Advanced_CV.git"   # push your branch first
BRANCH = "main"
import os, subprocess, sys
os.chdir("/kaggle/working")
url = REPO_URL
try:  # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("DA3_LAB_Advanced_CV"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "DA3_LAB_Advanced_CV"], check=True)
else:
    subprocess.run(["git", "-C", "DA3_LAB_Advanced_CV", "pull", "--ff-only"], check=True)
os.chdir("/kaggle/working/DA3_LAB_Advanced_CV")
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
sys.path.insert(0, os.path.abspath("src"))   # the editable install is only seen by new processes, not this kernel
!git log -1 --format="commit %H  %cd"

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: right panel > Settings > Accelerator > GPU T4 (or P100), then run all again."
if True:
    !nvidia-smi -L
    _x = torch.randn(1, 4, 64, 32, device="cuda", dtype=torch.float16)   # fails here at once if this torch build
    torch.nn.functional.scaled_dot_product_attention(_x, _x, _x)         # does not support the GPU (not after the downloads)
    print("GPU OK:", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))

In [ ]:
# Open3D (TSDF meshing of the fountain) + self-test; without a working TSDF the fountain uses depth-grid meshing
import sys, json
if sys.version_info[:2] <= (3, 12):
    !pip install -q "open3d==0.19.0"
else:
    !pip install -q "open3d>=0.19.0"
!python scripts/check_open3d.py --out /kaggle/working/open3d_check.json
try:
    MESH = "tsdf" if json.load(open("/kaggle/working/open3d_check.json"))["legacy_tsdf_ok"] else "depth"
except Exception:
    MESH = "depth"
print("fountain mesh:", MESH)

### Configuration

In [ ]:
PRESET = "da3-small"     # primary model of the pre-registered plan
KITTI_RAW = "download"   # "download" = fetch the needed drives from KITTI (account + terms first; Internet on);
                         # or Copy path of attached KITTI zips / unpacked raw drives; "" = skip B, C, E, F
FOUNTAIN = ""            # folder with the fountain photos (Copy path); "" = skip D, H
RESUME_FROM = ""         # output of an earlier session of this notebook, e.g. "/kaggle/input/<name>/lora"

ARMS = ["e3", "pose_only", "own_output"]   # plan v2 arms (same data, steps, seeds)
SEEDS = [0, 1]                              # plan v2: 2 seeds
SEC6_DATASETS = ["hiroom", "7scenes", "dtu64"]
SEC6_RUNS = ["released", "e3_s0", "e3_s1"]  # runs of THIS session for part G (split them over sessions if needed)
SNAPSHOT_RUN = "e3_s0"                      # its snapshots (every 500 steps) are reconstructed in part H
IGNORE_GONOGO = False                       # True = train even without a GO (logged as a deviation)
RUN = {"check": True, "kitti": True, "gonogo": True, "fountain": True, "train": True,
       "kitti_test": True, "sec6": False, "fountain_arms": True, "analysis": True}
EIGEN_URL = "https://raw.githubusercontent.com/nianticlabs/monodepth2/master/splits/eigen/test_files.txt"
EIGEN_TRAIN_URL = "https://raw.githubusercontent.com/nianticlabs/monodepth2/master/splits/eigen_full/train_files.txt"
SESSION_H = 11.5         # Kaggle stops a session at 12 h: training stops cleanly (resumable) before that
OUT = "/kaggle/working/lora"
RESULTS = f"{OUT}/results"                 # standard result records (copy into experiments/results/)
CANON = "/tmp/canonical/kitti"             # converted drives (not saved: redone after a resume)
RAW_TMP = "/tmp/kitti_raw"                 # unpacked KITTI zips, if the input holds zips
CACHE = "/tmp/lora_cache"                  # teacher / frozen outputs of the training clips (rebuilt if lost)
BENCH = "/tmp/da3bench"
WORK = "/tmp/eval_lora"                    # harness exports of part G
import os, shutil, time
T_START = time.time()
def session_hours_left():
    return SESSION_H - (time.time() - T_START) / 3600 - 1.0   # 1 h kept for parts F-I and saving
os.makedirs(OUT, exist_ok=True)
if RESUME_FROM:
    !cp -rn {RESUME_FROM}/. {OUT}/ && ls {OUT}
def save_checkpoint():      # Kaggle: the saved version output is the checkpoint
    pass
def free_kitti_checkpoint():
    pass
def free_work(tag):
    shutil.rmtree(f"{WORK}/workspace/evaluation/{tag}", ignore_errors=True)
for name, path in (("KITTI_RAW", KITTI_RAW), ("FOUNTAIN", FOUNTAIN)):
    if not path:
        print(f"{name} is empty: its parts are skipped")

In [ ]:
# --- shared helpers (resumable parts) -----------------------------------------------------------
import glob, json, os, time, urllib.request, zipfile
import torch
_exit_code = None   # set by every "!" command (IPython); 0 = success
os.makedirs(f"{OUT}/.done", exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
def is_done(path):
    """A finished step leaves a complete JSON file (a truncated copy does not count)."""
    try:
        json.load(open(path)); return True
    except Exception:
        return False
def done(step):
    return os.path.exists(f"{OUT}/.done/{step}")
def mark_done(step):
    open(f"{OUT}/.done/{step}", "w").close()
def run_dir(name):            # "e3_s0" -> runs/<preset>_e3_s0
    return f"{OUT}/runs/{PRESET}_{name}"
def adapter_path(name):
    return f"{run_dir(name)}/adapter.pt"
def trained():
    return [f"{a}_s{s}" for s in SEEDS for a in ARMS if is_done(f"{run_dir(f'{a}_s{s}')}/train_summary.json")]
def prepare_raw(src):
    """KITTI input -> folder with unpacked drives. Zips under `src` are extracted once to RAW_TMP."""
    zips = sorted(glob.glob(f"{src}/**/*.zip", recursive=True))
    if not zips:
        return src
    os.makedirs(RAW_TMP, exist_ok=True)
    for z in zips:
        mark = f"{RAW_TMP}/.unzipped_{os.path.basename(z)}"
        if not os.path.exists(mark):
            print("unpacking", os.path.basename(z), flush=True)
            zipfile.ZipFile(z).extractall(RAW_TMP)
            open(mark, "w").close()
    return RAW_TMP
def log_deviation(text):
    with open(f"{OUT}/deviations.txt", "a") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M ") + text + "\n")
    print("[deviation logged]", text)
GPU_MSG = "This part runs the model and needs a GPU (see the first cell)."
KITTI_TEST = f"{OUT}/kitti_test_{PRESET}.json"
GONOGO = f"{OUT}/gonogo_{PRESET}.json"
SEC6_TAG = lambda name: f"lora_sec6_{PRESET}_{name}_{'-'.join(SEC6_DATASETS)}"
SEC6_REC = lambda name: f"{RESULTS}/{'baseline' if name == 'released' else 'lora'}/{SEC6_TAG(name)}.json"
print("A check:", "DONE" if done("check") else "to do", "| B KITTI:", "DONE" if done("kitti") else "to do",
      "| C go/no-go:", "DONE" if is_done(GONOGO) else "to do", "| D fountain:", "DONE" if done(f"fountain_{PRESET}") else "to do")
print("E trained:", trained() or "none", "| F KITTI test:", "DONE" if is_done(KITTI_TEST) else "to do",
      "| G Sec. 6 done:", [n for n in ["released"] + [f"e3_s{s}" for s in SEEDS] if is_done(SEC6_REC(n))] or "none")

Part G setup (harness, Open3D check, Python 3.12 scoring environment): runs only if `RUN["sec6"]`

In [ ]:
if RUN["sec6"]:
    # --- official DA3 repo (reference / parity oracle / benchmark harness), pinned commit ----------
    import os, subprocess
    if not os.path.exists("/kaggle/working/Depth-Anything-3"):
        subprocess.run(["git", "clone", "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
                        "/kaggle/working/Depth-Anything-3"], check=True)
    subprocess.run(["git", "-C", "/kaggle/working/Depth-Anything-3", "checkout", "3d835ec1a5802d64a8b8b15f817a1ab54809bfe4"], check=True)
    OFFICIAL_SRC = "/kaggle/working/Depth-Anything-3/src"
    !pip install -q -r requirements-eval.txt
    # Open3D: requirements-eval installs the newest one; 0.20.0's TSDF returns empty meshes (isl-org/Open3D#7592),
    # so use the verified 0.19.0 where it exists (Python <= 3.12)
    import sys
    if sys.version_info[:2] <= (3, 12):
        !pip install -q "open3d==0.19.0"
    # Open3D self-test: does TSDF fusion (depth maps -> mesh) work in this environment?
    !python scripts/check_open3d.py --out /kaggle/working/open3d_check.json

In [ ]:
if RUN["sec6"]:
    import json
    chk = json.load(open("/kaggle/working/open3d_check.json"))
    RECON = bool(chk["legacy_tsdf_ok"])   # reconstruction scores (F1 / Chamfer) need a working Open3D TSDF
    if RECON:
        print("Open3D TSDF OK:", chk["open3d"], "-> reconstruction scores (F1 / Chamfer) included")
    else:
        print("Open3D", chk["open3d"], "on Python", chk["python"], "has a broken TSDF fusion (known Open3D 0.20 bug, "
              "github.com/isl-org/Open3D/issues/7592; the working 0.19.0 only exists up to Python 3.12).")
        print("This run SKIPS the reconstruction scores (F1 / Chamfer). Pose accuracy (AUC), failure detection "
              "and depth uncertainty are not affected.")
    MODES = "pose recon_unposed" if RECON else "pose"
    RECON_VARIANTS = "" if RECON else "--recon-variants"   # an empty list = pose-only harness replays

In [ ]:
if RUN["sec6"]:
    # --- reconstruction scores: Python 3.12 side environment (only when the Open3D above is broken) --------
    # Kaggle / Colab run Python 3.13, whose only Open3D (0.20.0) has a broken TSDF fusion (isl-org/Open3D#7592),
    # and the working 0.19.0 needs Python <= 3.12. The scoring (CPU only: TSDF fusion, F1 / Chamfer, pose AUC)
    # then runs in a small Python 3.12 environment; DA3 itself still runs on the GPU in this notebook's Python.
    import json, os
    PY312 = "/tmp/py312/bin/python"
    SPLIT = False   # True = model in this Python (--infer-only), scores in PY312 (--eval-only)
    if not RECON:
        !pip install -q uv
        !uv python install 3.12
        if not os.path.exists(PY312):
            !uv venv /tmp/py312 --python 3.12
        !uv pip install -q --python {PY312} torch torchvision --index-url https://download.pytorch.org/whl/cpu
        !uv pip install -q --python {PY312} numpy scipy matplotlib "open3d==0.19.0" "evo==1.38.1" addict scikit-learn tqdm omegaconf PyYAML trimesh plyfile einops safetensors huggingface_hub opencv-python-headless pillow imageio "moviepy<2"
        !uv pip install -q --python {PY312} --no-deps -e .
        !{PY312} scripts/check_open3d.py --out /tmp/open3d_check_py312.json
        try:
            SPLIT = bool(json.load(open("/tmp/open3d_check_py312.json"))["legacy_tsdf_ok"])
        except Exception as e:
            print("Python 3.12 scoring environment not usable:", repr(e))
        if SPLIT:
            MODES, RECON_VARIANTS = "pose recon_unposed", ""
            print("OK: the reconstruction scores (F1 / Chamfer) are computed with Python 3.12 + Open3D 0.19.0.")
        else:
            print("The Python 3.12 scoring environment does not work either: this run stays POSE ONLY. "
                  "Send the output of this cell.")
    SCORER = PY312 if SPLIT else "python"   # the Python that scores (and runs the analyses)

### A · Platform check (unit tests + released weights on the GPU)

In [ ]:
if RUN["check"]:
    if done("check"):
        print("[checkpoint] part A already complete, skipped")
    else:
        assert torch.cuda.is_available(), GPU_MSG
        !python -m pytest -q tests/test_lora.py tests/test_kitti.py tests/test_lora_train.py tests/test_lora_stats.py 2>&1 | tail -n 3
        !python scripts/lora_check.py --presets {PRESET} --out {OUT}/lora_check.json
        if _exit_code == 0 and is_done(f"{OUT}/lora_check.json"):
            mark_done("check")
        save_checkpoint()
    if is_done(f"{OUT}/lora_check.json"):
        chk = json.load(open(f"{OUT}/lora_check.json"))
        for p, r in chk["bit"].items():
            print(p, "| deterministic repeat:", r["deterministic"], "| LoRA off / B=0 bit-identical:", r["pass"],
                  "| trainable LoRA params:", r["lora_params"])
        for k, r in chk["probe"].items():
            print(f"{k:28s}", f"peak {r['peak_gb']:.2f} GB, {r['s_per_step']:.2f} s/step" if r["ok"] else "OUT OF MEMORY")

### B · KITTI raw -> canonical format + drive split

In [ ]:
KITTI_OK = is_done(f"{CANON}/split.json") and done("kitti")
if RUN["kitti"] and KITTI_RAW:
    if KITTI_OK:
        print("[checkpoint] part B already complete, skipped")
    else:
        os.makedirs(f"{OUT}/kitti", exist_ok=True)
        urllib.request.urlretrieve(EIGEN_URL, f"{OUT}/kitti_eigen_test_files.txt")
        SPLIT_FROM = ""
        if KITTI_RAW == "download":   # official KITTI servers (register at cvlibs.net and accept the terms first)
            urllib.request.urlretrieve(EIGEN_TRAIN_URL, f"{OUT}/kitti_eigen_train_files.txt")
            os.makedirs(RAW_TMP, exist_ok=True)
            if os.path.exists(f"{OUT}/kitti/planned_split.json"):   # keep the split of an earlier session
                !cp -n {OUT}/kitti/planned_split.json {RAW_TMP}/planned_split.json
            !python scripts/download_kitti.py --out {RAW_TMP} --eigen-test-list {OUT}/kitti_eigen_test_files.txt                 --eigen-train-list {OUT}/kitti_eigen_train_files.txt
            !cp {RAW_TMP}/planned_split.json {OUT}/kitti/planned_split.json
            RAW, SPLIT_FROM = RAW_TMP, f"--split-from {RAW_TMP}/planned_split.json"
        else:
            RAW = prepare_raw(KITTI_RAW)
        !python scripts/prepare_kitti.py --raw "{RAW}" --out {CANON} --eigen-test-list {OUT}/kitti_eigen_test_files.txt --skip-existing {SPLIT_FROM}
        if _exit_code == 0 and is_done(f"{CANON}/split.json"):
            os.makedirs(f"{OUT}/kitti", exist_ok=True)
            !cp {CANON}/split.json {OUT}/kitti/split.json
            mark_done("kitti")
        save_checkpoint()
    KITTI_OK = is_done(f"{CANON}/split.json")
if KITTI_OK:
    split = json.load(open(f"{CANON}/split.json"))
    print("train:", split["train"], "\nval (go/no-go):", split["val"], "\ntest (held out):", split["test"])
elif RUN["kitti"]:
    print("no converted KITTI drives: set KITTI_RAW (parts B, C, E, F need them)")

### C · Go/no-go: does the pose-conditioned DA3 (E3's teacher) give better depth than the unposed one?
Validation drives only. 8 frames (every 3rd), 10 clips per drive, 1.8:1 centre crop at 504x280,
depth vs LiDAR with one median scale per clip, cameras from the camera head. **GO if posed / unposed
AbsRel ≤ 0.95** (report §4.4). If only ATE / RPE improve, E3 reduces to pose-supervised fine-tuning.

In [ ]:
if RUN["gonogo"] and KITTI_OK:
    if is_done(GONOGO):
        print("[checkpoint] part C already complete, skipped")
    else:
        assert torch.cuda.is_available(), GPU_MSG
        !python scripts/kitti_eval.py --root {CANON} --drives val --arms released posed --preset {PRESET} --out {GONOGO}
        save_checkpoint()
if is_done(GONOGO):
    from IPython.display import Markdown, display
    display(Markdown(open(GONOGO.replace(".json", ".md")).read()))

### D · Fountain: reference reconstruction with the released weights
Every adapter and training snapshot is later run on the same photos, order and resolution (part H).
No ground truth: it shows how the reconstruction *changes*, never whether it is better.

In [ ]:
FREF = f"{OUT}/fountain/released_{PRESET}"
if RUN["fountain"] and FOUNTAIN:
    if done(f"fountain_{PRESET}"):
        print("[checkpoint] part D already complete, skipped")
    else:
        assert torch.cuda.is_available(), GPU_MSG
        !python scripts/infer.py --images "{FOUNTAIN}" --out {FREF} --weights official:{PRESET} --mesh {MESH}
        if _exit_code == 0:
            !python scripts/render_preview.py {FREF}
            mark_done(f"fountain_{PRESET}")
        save_checkpoint()
if done(f"fountain_{PRESET}"):
    from IPython.display import Image, display
    for f in ("preview.png", "preview_3d.png"):
        if os.path.exists(f"{FREF}/{f}"):
            display(Image(f"{FREF}/{f}", width=1000))

### E · Training (LoRA on blocks 4-11, recipe frozen in `configs/lora_analysis.yaml`)
Arms × seeds, one complete seed after the other, so a cut session still leaves whole seeds. The cache
(teacher + frozen outputs of the training clips) is built once and shared. Each run saves an
adapter-only file, snapshots every 500 steps and a resumable checkpoint; finished runs are skipped.
Pre-registered: E3 is trained only after a **GO** (or with `IGNORE_GONOGO = True`, which is logged as a
deviation in `deviations.txt`).

In [ ]:
TRAIN_OK = False
if RUN["train"] and KITTI_OK:
    gate = json.load(open(GONOGO))["summary"]["gate"]["decision"] if is_done(GONOGO) else None
    if gate == "GO":
        TRAIN_OK = True
    elif IGNORE_GONOGO:
        TRAIN_OK = True
        if not done("deviation_gonogo"):
            log_deviation(f"training run although the go/no-go was {gate or 'not run'} (IGNORE_GONOGO=True)")
            mark_done("deviation_gonogo")
    else:
        print(f"go/no-go = {gate or 'not run'}: training skipped, as pre-registered "
              "(IGNORE_GONOGO = True trains anyway and logs a deviation)")
if TRAIN_OK:
    for seed in SEEDS:
        for arm in ARMS:
            name = f"{arm}_s{seed}"
            if is_done(f"{run_dir(name)}/train_summary.json"):
                print(f"[checkpoint] {name}: trained, skipped"); continue
            budget = session_hours_left()
            if budget < 0.25:
                print(f"session time almost used up ({budget:.2f} h left): stopping before {name}; "
                      "save the version and resume (RESUME_FROM)"); break
            assert torch.cuda.is_available(), GPU_MSG
            !python scripts/train_lora.py --root {CANON} --arm {arm} --seed {seed} --preset {PRESET} \
                --cache {CACHE} --out {OUT}/runs --resume --time-budget-h {budget:.2f}
            save_checkpoint()
        else:
            continue
        break
print("trained:", trained())

### F · Held-out KITTI test drives: released vs every trained adapter (identical clips)
The test drives are evaluated once per adapter set; the result feeds the pre-registered H1 / S1-S5.

In [ ]:
if RUN["kitti_test"] and KITTI_OK:
    names = trained()
    have = set(json.load(open(KITTI_TEST))["adapters"]) if is_done(KITTI_TEST) else None
    if not split["test"]:
        print("no Eigen test drive converted: add Eigen test drives to the KITTI input")
    elif not names:
        print("no trained adapter yet (part E)")
    elif have is not None and have >= set(names):
        print("[checkpoint] part F already complete for", sorted(have), "- skipped")
    else:
        assert torch.cuda.is_available(), GPU_MSG
        ads = " ".join(f"{n}={adapter_path(n)}" for n in names)
        !python scripts/kitti_eval.py --root {CANON} --drives test --arms released --adapters {ads} --preset {PRESET} --out {KITTI_TEST}
        save_checkpoint()
        free_kitti_checkpoint()
if is_done(KITTI_TEST):
    from IPython.display import Markdown, display
    display(Markdown(open(KITTI_TEST.replace(".json", ".md")).read()))

### G · Sec. 6 no-regression: released vs E3 adapters, the paper's protocol (released harness)
HiRoom, 7Scenes, DTU-64; pose AUC + TSDF reconstruction (Python 3.12 side environment when this
Open3D's TSDF is broken). `SEC6_RUNS` chooses the runs of this session (e.g. `["released"]` here and
`["e3_s0", "e3_s1"]` on the other platform); finished runs are skipped, records combine.

In [ ]:
if RUN["sec6"]:
    from huggingface_hub import hf_hub_download
    import psutil
    FUSION_WORKERS = max(1, min(os.cpu_count() or 1, int(psutil.virtual_memory().total / 2**30 // 6), 8))
    os.makedirs(BENCH, exist_ok=True)
    for d in SEC6_DATASETS:
        if not os.path.exists(f"{BENCH}/{d}"):
            z = hf_hub_download("depth-anything/DA3-BENCH", f"{d}.zip", repo_type="dataset", local_dir=f"{BENCH}_zips")
            zipfile.ZipFile(z).extractall(BENCH); os.remove(z)
    DS6 = " ".join(SEC6_DATASETS)
    for name in SEC6_RUNS:
        t, rec = SEC6_TAG(name), SEC6_REC(name)
        if is_done(rec):
            print(f"[checkpoint] {name}: Sec. 6 record exists, skipped"); continue
        if name != "released" and not os.path.exists(adapter_path(name)):
            print(f"{name}: no adapter (train it in part E or resume from that output)"); continue
        enh, var = ("baseline", "head") if name == "released" else ("lora", name)
        ad = "" if name == "released" else f"--adapter {adapter_path(name)} --baseline-tag {SEC6_TAG('released')}"
        common = (f"--official-src {OFFICIAL_SRC} --bench-root {BENCH} --work-root {WORK} --weights official:{PRESET} "
                  f"--datasets {DS6} --modes {MODES} --tag {t} {ad}")
        if SPLIT:   # model here (GPU), scores in the Python 3.12 environment (CPU)
            if not os.path.exists(f"{WORK}/.done_infer_{t}"):
                assert torch.cuda.is_available(), GPU_MSG
                !python scripts/eval_bench.py {common} --infer-only --skip-existing
                if _exit_code == 0:
                    open(f"{WORK}/.done_infer_{t}", "w").close()
                save_checkpoint()
            !{PY312} scripts/eval_bench.py {common} --eval-only --enhancement {enh} --variant {var} \
                --results-root {RESULTS} --fusion-workers {FUSION_WORKERS}
        else:
            assert torch.cuda.is_available(), GPU_MSG
            !python scripts/eval_bench.py {common} --enhancement {enh} --variant {var} --results-root {RESULTS} \
                --fusion-workers {FUSION_WORKERS} --skip-existing
        save_checkpoint()
        if is_done(rec):
            free_work(t)
            print(f"{name}: done ->", rec)

### H · Fountain: every adapter and the training snapshots on the same photos
Differences to the released reconstruction (depth, cameras, Chamfer / F1 between the clouds), a
side-by-side orbit video with one camera path and aligned PLYs for Blender (`compare_*/aligned/`).
Snapshot `step_00000` has B = 0, so it must equal the released model (a visible sanity check).

In [ ]:
if RUN["fountain_arms"] and FOUNTAIN and done(f"fountain_{PRESET}"):
    models = {n: adapter_path(n) for n in trained()}
    snaps = {f"{SNAPSHOT_RUN}_{os.path.basename(p)[:-3]}": p
             for p in sorted(glob.glob(f"{run_dir(SNAPSHOT_RUN)}/snapshots/step_*.pt"))}
    for name, path in {**models, **snaps}.items():
        fo = f"{OUT}/fountain/{name}"
        if done(f"fountain_{PRESET}_{name}"):
            continue
        assert torch.cuda.is_available(), GPU_MSG
        !python scripts/infer.py --images "{FOUNTAIN}" --out {fo} --weights official:{PRESET} --adapter {path} --mesh {MESH} > /dev/null
        if _exit_code == 0:
            mark_done(f"fountain_{PRESET}_{name}")
        save_checkpoint()
    from IPython.display import Markdown, display
    for label, group in (("final", models), ("snapshots", snaps)):
        tests = " ".join(f"{n}={OUT}/fountain/{n}" for n in group if done(f"fountain_{PRESET}_{n}"))
        if tests:
            !python scripts/compare_reconstructions.py --ref {FREF} --test {tests} --out {OUT}/fountain/compare_{label} > /dev/null
            display(Markdown(f"**{label}**\n\n" + open(f"{OUT}/fountain/compare_{label}/compare.md").read()))
            print("video:", f"{OUT}/fountain/compare_{label}/side_by_side.mp4")
    save_checkpoint()
elif RUN["fountain_arms"]:
    print("part H needs FOUNTAIN and the reference of part D")

### I · Pre-registered analysis (plan v2)

In [ ]:
if RUN["analysis"] and is_done(KITTI_TEST):
    rel = SEC6_REC("released")
    e3 = " ".join(SEC6_REC(f"e3_s{s}") for s in SEEDS if is_done(SEC6_REC(f"e3_s{s}")))
    sec6 = (f"--sec6-released {rel} --sec6-e3 {e3}") if is_done(rel) and e3 else ""
    !python scripts/lora_analysis.py --kitti {KITTI_TEST} {sec6} --out {OUT}/analysis --results-root {RESULTS}
    save_checkpoint()
    from IPython.display import Markdown, display
    if os.path.exists(f"{OUT}/analysis/summary.md"):
        display(Markdown(open(f"{OUT}/analysis/summary.md").read()))
elif RUN["analysis"]:
    print("part I needs the KITTI test evaluation (part F)")

### Save: download `lora_results.zip` (Output panel); copy `lora/results/*` into the repo's `experiments/results/` and commit

In [ ]:
!cd /kaggle/working && rm -f lora_results.zip && zip -qr lora_results.zip lora -x "lora/fountain/*/ray/*" && ls -la lora_results.zip
!find {OUT} -maxdepth 3 \( -name "*.json" -o -name "*.md" -o -name "*.mp4" \) -not -path "*/fountain/*/*/*" | sort | head -50